# 04. CReM baseline

**Baseline**: context-compatible local replacement using CReM, with Murcko scaffold
atoms protected via `protected_ids`.

Design:
- `protected_ids` = Murcko scaffold atom indices → scaffold atoms never replaced
- Scaffold retention additionally confirmed with `scaffold_is_retained()` (shared with all methods)
- No descriptor information used during generation; evaluated against all 7 tasks afterwards
- `candidate_rank` = order of accepted candidates (invalid/scaffold_failed/duplicate excluded)

Generation statistics use exclusive categories:
```
n_examined = n_invalid + n_scaffold_failed + n_duplicate + n_valid_unique
```
Processing order: attempted → valid? → scaffold_retained? → not_duplicate? → accepted


In [ ]:
import sys, traceback
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")
from crem.crem import mutate_mol
sys.path.append("..")

from configs.benchmark_config import (
    ACTIVE_PARENT_DATASET, RESULTS_DIR,
    N_CANDIDATES_PER_PARENT, RANDOM_SEED, TASKS,
)
from utils.parents import prepare_parent
from utils.scaffold import get_scaffold_atom_ids, scaffold_is_retained
from utils.records import make_candidate_record

import os
os.makedirs(RESULTS_DIR, exist_ok=True)


## Parameters

| Parameter  | Value | Rationale |
|------------|-------|-----------|
| `radius`   | 3     | Standard CReM environment radius; balances context-specificity and coverage |
| `min_size` | 0     | Allow replacement with hydrogen (atom deletion equivalent) |
| `max_size` | 8     | Covers single-atom to small-ring replacements, comparable in scope to Scaffold-tuner's unified pool entries (which include fragments such as pyrrolyl and indolyl); avoids arbitrarily restricting CReM to single-atom edits |


In [ ]:
CREM_DB       = str(Path.home() / "software/CReM_data/chembl33_sa2_f5.db")
CREM_RADIUS   = 3
CREM_MIN_SIZE = 0
CREM_MAX_SIZE = 8
METHOD = "crem"


In [ ]:
def generate_crem_candidates(parent_mol, parent_smiles, scaffold_ids):
    """Return (candidates, stats_dict, errors_list).

    Generation statistics use explicit counters (no residual calculation):
        n_examined = n_invalid + n_scaffold_failed + n_duplicate + n_valid_unique

    Processing order:
        attempted → valid? → scaffold_retained? → not_duplicate? → accepted
    """
    seen = set([parent_smiles])
    candidates = []
    n_examined = 0; n_invalid = 0; n_scaffold_failed = 0; n_duplicate = 0
    errors = []

    try:
        for smi in mutate_mol(
            parent_mol,
            db_name=CREM_DB,
            radius=CREM_RADIUS,
            min_size=CREM_MIN_SIZE,
            max_size=CREM_MAX_SIZE,
            protected_ids=scaffold_ids,  # ← scaffold protection
            seed=RANDOM_SEED,
        ):
            if len(candidates) >= N_CANDIDATES_PER_PARENT:
                break
            n_examined += 1
            cand_mol = Chem.MolFromSmiles(smi)
            if cand_mol is None:
                n_invalid += 1          # invalid SMILES from CReM
                continue
            canon = Chem.MolToSmiles(cand_mol)
            if not scaffold_is_retained(parent_mol, cand_mol):
                n_scaffold_failed += 1  # valid but scaffold lost
                continue
            if canon in seen:
                n_duplicate += 1        # valid, scaffold-retained, duplicate/parent
                continue
            seen.add(canon)
            candidates.append(canon)

    except Exception as e:
        errors.append({
            "error_type":    type(e).__name__,
            "error_message": str(e),
            "traceback":     traceback.format_exc(),
        })

    n_valid_unique = len(candidates)
    assert n_invalid + n_scaffold_failed + n_duplicate + n_valid_unique == n_examined, (
        f"Stats sum mismatch: {n_invalid}+{n_scaffold_failed}+{n_duplicate}+{n_valid_unique} != {n_examined}"
    )

    stats = {
        "n_examined":       n_examined,
        "n_invalid":         n_invalid,
        "n_scaffold_failed": n_scaffold_failed,
        "n_duplicate":       n_duplicate,
        "n_valid_unique":    n_valid_unique,
        "crem_error":        len(errors) > 0,
    }
    return candidates, stats, errors


## Step 1: Smoke test on 5 parents

In [ ]:
df = pd.read_csv(ACTIVE_PARENT_DATASET)
if "canonical_smiles" not in df.columns and "smiles" in df.columns:
    df = df.rename(columns={"smiles": "canonical_smiles"})
if "parent_id" not in df.columns and "chembl_id" in df.columns:
    df = df.rename(columns={"chembl_id": "parent_id"})
print(f"Parents: {len(df)}")

records_test, stats_test, error_rows_test = [], [], []
for _, row in df.head(5).iterrows():
    prepared = prepare_parent(row)
    if prepared is None:
        continue
    parent_id, parent_smiles, parent_desc, _ = prepared
    parent_mol = Chem.MolFromSmiles(parent_smiles)
    scaffold_ids = get_scaffold_atom_ids(parent_mol)

    candidates, stats, errors = generate_crem_candidates(
        parent_mol, parent_smiles, scaffold_ids
    )
    stats.update({"method": METHOD, "parent_id": parent_id})
    stats_test.append(stats)
    for e in errors:
        error_rows_test.append({"parent_id": parent_id, **e})

    for task_name in TASKS:
        for rank, cand_smi in enumerate(candidates, start=1):
            rec = make_candidate_record(
                method=METHOD, parent_id=parent_id,
                parent_smiles=parent_smiles, task_name=task_name,
                candidate_rank=rank, candidate_smiles=cand_smi,
                parent_desc=parent_desc,
            )
            if rec is not None:
                records_test.append(rec)

df_test = pd.DataFrame(records_test)
print(f"Records: {len(df_test)}  Tasks: {df_test['task'].nunique() if len(df_test) else 0}/7")
if len(df_test):
    print(f"scaffold_retained all True: {df_test['scaffold_retained'].all()}")
pd.DataFrame(stats_test)


## Step 2: Full run — 1000 parents

In [ ]:
records, gen_stats, error_rows = [], [], []

for _, row in tqdm(df.iterrows(), total=len(df), desc="CReM"):
    prepared = prepare_parent(row)
    if prepared is None:
        continue
    parent_id, parent_smiles, parent_desc, _ = prepared
    parent_mol = Chem.MolFromSmiles(parent_smiles)
    scaffold_ids = get_scaffold_atom_ids(parent_mol)

    candidates, stats, errors = generate_crem_candidates(
        parent_mol, parent_smiles, scaffold_ids
    )
    stats.update({"method": METHOD, "parent_id": parent_id})
    gen_stats.append(stats)
    for e in errors:
        error_rows.append({"parent_id": parent_id, **e})

    for task_name in TASKS:
        for rank, cand_smi in enumerate(candidates, start=1):
            rec = make_candidate_record(
                method=METHOD, parent_id=parent_id,
                parent_smiles=parent_smiles, task_name=task_name,
                candidate_rank=rank, candidate_smiles=cand_smi,
                parent_desc=parent_desc,
            )
            if rec is not None:
                records.append(rec)

print(f"Total records: {len(records)}")
print(f"Parents with CReM errors: {sum(r['crem_error'] for r in gen_stats)}")


In [ ]:
OUTPUT_FILE = f"{RESULTS_DIR}/crem.csv"
STATS_FILE  = f"{RESULTS_DIR}/crem_generation_stats.csv"
ERROR_FILE  = f"{RESULTS_DIR}/crem_errors.csv"
pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
pd.DataFrame(gen_stats).to_csv(STATS_FILE, index=False)
pd.DataFrame(error_rows).to_csv(ERROR_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")
print(f"Saved → {STATS_FILE}")
print(f"Saved → {ERROR_FILE} ({len(error_rows)} errors)")
